# Lab 2: House Segmentation from Aerial Imagery
**SEG4180 Applied Machine Learning — Stuti Pandya**

Goal: Train a UNet segmentation model on aerial satellite images to detect houses.  
Dataset: [Inria Aerial Image Labeling Dataset](https://huggingface.co/datasets/jlbaker361/inria-aerial-labeling) via HuggingFace  
Pixel mask generation: SAM (Segment Anything Model) approach from Week 7  
Metrics: IoU (Jaccard Index) and Dice Score

## 1. Install Dependencies

In [ ]:
!pip install -q datasets segment-anything huggingface_hub torch torchvision
!pip install -q matplotlib numpy Pillow scikit-learn tqdm

## 2. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 3. Load Dataset from HuggingFace

In [ ]:
# Load the Inria Aerial Image Labeling dataset
# This dataset contains 360x360 aerial image patches with building masks
print('Loading dataset...')
ds = load_dataset('jlbaker361/inria-aerial-labeling', split='train')
print(f'Total samples: {len(ds)}')
print(f'Features: {ds.features}')

# Preview a sample
sample = ds[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample['image'])
axes[0].set_title('Aerial Image')
axes[1].imshow(sample['label'], cmap='gray')
axes[1].set_title('Ground Truth Mask')
plt.tight_layout()
plt.savefig('sample_preview.png', dpi=100)
plt.show()
print('Mask unique values:', np.unique(np.array(sample['label'])))

## 4. Pixel Mask Generation (Week 7 SAM Approach)

In [ ]:
# Week 7 approach: Use SAM to generate pixel-level candidate masks,
# then filter by overlap with ground truth building labels.
# This mirrors the week7-segment_houses_smartly.ipynb notebook.

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import urllib.request

# Download SAM checkpoint (ViT-B for speed on Kaggle)
SAM_CHECKPOINT = 'sam_vit_b_01ec64.pth'
if not os.path.exists(SAM_CHECKPOINT):
    print('Downloading SAM checkpoint...')
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        SAM_CHECKPOINT
    )
    print('Download complete.')

sam = sam_model_registry['vit_b'](checkpoint=SAM_CHECKPOINT)
sam.to(device)
mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=16,          # fewer points = faster generation
    pred_iou_thresh=0.88,
    stability_score_thresh=0.92,
    min_mask_region_area=100,
)
print('SAM loaded.')

In [ ]:
def generate_sam_building_mask(image_pil, gt_mask_pil, iou_threshold=0.3):
    """
    Week 7 pixel mask generation approach:
    1. Run SAM to get many candidate masks
    2. Keep only masks that overlap significantly with ground truth building labels
    3. Merge kept masks into a single binary mask
    """
    image_np = np.array(image_pil.convert('RGB'))
    gt_np = np.array(gt_mask_pil)
    # Normalize ground truth to binary (buildings = 1)
    gt_binary = (gt_np > 127).astype(np.uint8)

    # Generate SAM masks
    sam_masks = mask_generator.generate(image_np)

    # Filter: keep SAM masks that agree with ground truth
    final_mask = np.zeros(gt_binary.shape, dtype=np.uint8)
    for m in sam_masks:
        seg = m['segmentation'].astype(np.uint8)
        intersection = np.logical_and(seg, gt_binary).sum()
        union = np.logical_or(seg, gt_binary).sum()
        iou = intersection / (union + 1e-6)
        if iou >= iou_threshold:
            final_mask = np.logical_or(final_mask, seg).astype(np.uint8)

    return final_mask


# Demo on one sample
sample = ds[5]
print('Generating SAM mask for sample...')
sam_mask = generate_sam_building_mask(sample['image'], sample['label'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sample['image'])
axes[0].set_title('Aerial Image')
axes[1].imshow(np.array(sample['label']), cmap='gray')
axes[1].set_title('Ground Truth Mask')
axes[2].imshow(sam_mask, cmap='gray')
axes[2].set_title('SAM-Generated Mask')
plt.tight_layout()
plt.savefig('sam_mask_demo.png', dpi=100)
plt.show()

## 5. Dataset Preparation

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 8
NUM_SAMPLES = 500  # Use subset for reasonable Kaggle training time

# Use ground truth masks directly for training
# (SAM masks are used for analysis/demo above, GT labels are cleaner for training)
indices = list(range(min(NUM_SAMPLES, len(ds))))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')


class BuildingDataset(Dataset):
    def __init__(self, hf_dataset, indices, img_size=256, augment=False):
        self.ds = hf_dataset
        self.indices = indices
        self.img_size = img_size
        self.augment = augment

        self.img_transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.mask_transform = T.Compose([
            T.Resize((img_size, img_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        sample = self.ds[self.indices[idx]]
        image = sample['image'].convert('RGB')
        mask = sample['label'].convert('L')

        # Augmentation (flip + rotation) — applied consistently to image and mask
        if self.augment:
            if torch.rand(1) > 0.5:
                image = T.functional.hflip(image)
                mask = T.functional.hflip(mask)
            if torch.rand(1) > 0.5:
                image = T.functional.vflip(image)
                mask = T.functional.vflip(mask)
            angle = torch.randint(-30, 30, (1,)).item()
            image = T.functional.rotate(image, angle)
            mask = T.functional.rotate(mask, angle)

        image = self.img_transform(image)
        mask = self.mask_transform(mask)
        mask = (mask > 0.5).float()  # binarize
        return image, mask


train_dataset = BuildingDataset(ds, train_idx, IMG_SIZE, augment=True)
val_dataset   = BuildingDataset(ds, val_idx,   IMG_SIZE, augment=False)
test_dataset  = BuildingDataset(ds, test_idx,  IMG_SIZE, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('DataLoaders ready.')

## 6. UNet Model Architecture

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive Conv2d + BN + ReLU blocks."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """Standard UNet for binary semantic segmentation."""
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2, 2)

        # Encoder
        for feat in features:
            self.downs.append(DoubleConv(in_channels, feat))
            in_channels = feat

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder
        for feat in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feat * 2, feat, 2, 2))
            self.ups.append(DoubleConv(feat * 2, feat))

        self.final_conv = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skip_connections = []

        # Encoder path
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        # Decoder path
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skip_connections[i // 2]
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i + 1](x)

        return self.final_conv(x)


model = UNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'UNet parameters: {total_params:,}')

## 7. Loss Function and Metrics

In [ ]:
class DiceLoss(nn.Module):
    """Dice loss for binary segmentation."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred_flat   = pred.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        return 1 - (2. * intersection + self.smooth) / (
            pred_flat.sum() + target_flat.sum() + self.smooth
        )


class CombinedLoss(nn.Module):
    """BCE + Dice combined loss — standard for segmentation."""
    def __init__(self):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):
        return self.bce(pred, target) + self.dice(pred, target)


def compute_iou(pred_logits, target, threshold=0.5):
    """Jaccard Index (IoU) for binary masks."""
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - intersection
    return (intersection / (union + 1e-6)).mean().item()


def compute_dice(pred_logits, target, threshold=0.5):
    """Dice score for binary masks."""
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    intersection = (pred * target).sum(dim=(1, 2, 3))
    return (2 * intersection / (pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + 1e-6)).mean().item()


criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)
print('Loss, metrics, and optimizer ready.')

## 8. Training Loop

In [ ]:
NUM_EPOCHS = 20
best_val_iou = 0
history = {'train_loss': [], 'val_loss': [], 'train_iou': [], 'val_iou': [], 'train_dice': [], 'val_dice': []}

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    train_loss, train_iou, train_dice = 0, 0, 0

    for images, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]'):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_iou  += compute_iou(outputs, masks)
        train_dice += compute_dice(outputs, masks)

    train_loss /= len(train_loader)
    train_iou  /= len(train_loader)
    train_dice /= len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss, val_iou, val_dice = 0, 0, 0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, masks).item()
            val_iou  += compute_iou(outputs, masks)
            val_dice += compute_dice(outputs, masks)

    val_loss /= len(val_loader)
    val_iou  /= len(val_loader)
    val_dice /= len(val_loader)

    scheduler.step(val_loss)

    # Save best model
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), 'best_unet.pth')

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_iou'].append(train_iou)
    history['val_iou'].append(val_iou)
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_dice)

    print(f'Epoch {epoch+1:02d} | '
          f'Train Loss: {train_loss:.4f} IoU: {train_iou:.4f} Dice: {train_dice:.4f} | '
          f'Val Loss: {val_loss:.4f} IoU: {val_iou:.4f} Dice: {val_dice:.4f}')

print(f'\nBest Val IoU: {best_val_iou:.4f}')

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'],   label='Validation')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Combined Loss (BCE + Dice)')
axes[0].legend()

axes[1].plot(epochs_range, history['train_iou'], label='Train')
axes[1].plot(epochs_range, history['val_iou'],   label='Validation')
axes[1].set_title('IoU Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Jaccard Index (IoU)')
axes[1].legend()

axes[2].plot(epochs_range, history['train_dice'], label='Train')
axes[2].plot(epochs_range, history['val_dice'],   label='Validation')
axes[2].set_title('Dice Score Curve')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Dice Score')
axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 10. Evaluation on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_unet.pth', map_location=device))
model.eval()

test_iou, test_dice = 0, 0
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        test_iou  += compute_iou(outputs, masks)
        test_dice += compute_dice(outputs, masks)

test_iou  /= len(test_loader)
test_dice /= len(test_loader)

print('=' * 40)
print(f'TEST SET RESULTS')
print(f'  IoU (Jaccard): {test_iou:.4f}')
print(f'  Dice Score:    {test_dice:.4f}')
print('=' * 40)

## 11. Visualize Predictions

In [ ]:
# Denormalize helper
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denorm(t):
    return torch.clamp(t * std + mean, 0, 1)

model.eval()
images_batch, masks_batch = next(iter(test_loader))
images_batch = images_batch.to(device)

with torch.no_grad():
    preds = torch.sigmoid(model(images_batch))

n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))

for i in range(n_show):
    img_vis  = denorm(images_batch[i].cpu()).permute(1, 2, 0).numpy()
    gt_vis   = masks_batch[i].squeeze().numpy()
    pred_vis = (preds[i].squeeze().cpu().numpy() > 0.5).astype(float)

    axes[i, 0].imshow(img_vis)
    axes[i, 0].set_title('Aerial Image')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(gt_vis, cmap='gray')
    axes[i, 1].set_title('Ground Truth Mask')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(pred_vis, cmap='gray')
    axes[i, 2].set_title('Predicted Mask')
    axes[i, 2].axis('off')

plt.suptitle(f'UNet Predictions | Test IoU: {test_iou:.4f} | Test Dice: {test_dice:.4f}', fontsize=14)
plt.tight_layout()
plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved predictions.png')

## 12. Summary

| Metric | Score |
|--------|-------|
| Test IoU | See above |
| Test Dice | See above |

**Model**: UNet (encoder-decoder with skip connections)  
**Dataset**: Inria Aerial Image Labeling (~500 samples, 70/15/15 split)  
**Pixel mask generation**: SAM (Segment Anything) filtered by ground truth overlap — Week 7 approach  
**Augmentation**: horizontal/vertical flip, rotation ±30°  
**Loss**: BCE + Dice combined  
**Training**: 20 epochs, Adam optimizer, ReduceLROnPlateau